# Amazon ML Challange Hackathon

In [2]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import timm
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
import re

In [1]:
!nvidia-smi

Mon Oct 13 03:39:34 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.256.02   Driver Version: 470.256.02   CUDA Version: 12.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  On   | 00000000:C2:00.0 Off |                    0 |
| N/A   57C    P0    67W / 275W |      0MiB / 40536MiB |      0%      Default |
|                               |                      |             Disabled |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [3]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_linear_schedule_with_warmup
)
from torch.cuda.amp import autocast, GradScaler
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# This config was used till 20 epoches, and after we resumed our training from checkpoint and used the config in below code block

# class Config:
#     DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#     # --- MODIFICATION #1: REDUCE BATCH SIZE ---
#     # Larger models consume significantly more GPU memory (VRAM).
#     BATCH_SIZE = 32 # Reduced from 64. You might be able to increase to 32 later.
#     # ----------------------------------------

#     NUM_WORKERS = 8
#     TRAIN_SUBSET_FRACTION = 1.0
#     TEST_SUBSET_FRACTION = 1.0
#     EPOCHS = 30

#     # --- MODIFICATION #2: ADJUST LEARNING RATE ---
#     LR = 1e-5  # Reduced from 3e-5.
#     # ---------------------------------------------

#     MAX_LEN = 128

#     TEXT_MODEL = "microsoft/deberta-v3-large"

#     IMAGE_MODEL = "openai/clip-vit-large-patch14"  # For both the model and processor
#     # ---------------------------------------

#     TRAIN_IMAGE_DIR = "../images_train"
#     TEST_IMAGE_DIR = "../images_test"
#     SEED = 42

#     CHECKPOINT_DIR = "./checkpoints_deberta-v3-large_clip-large"
#     # ---------------------------------------------
    
#     LATEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")

# print(f"Using device: {Config.DEVICE}")
# print(f"Batch Size: {Config.BATCH_SIZE}, Num Workers: {Config.NUM_WORKERS}")
# print(f"Learning Rate: {Config.LR}")
# os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

Using device: cuda
Batch Size: 32, Num Workers: 8
Learning Rate: 1e-05


In [ ]:
# This code was run after epoch 20, resuming our training from checkpoint tell 30 epoch

class Config:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # --- FIX: Reduce physical batch size to prevent OOM on resume ---
    BATCH_SIZE = 16 # Reduced from 32
    # --- FIX: Add Gradient Accumulation to maintain effective batch size ---
    GRAD_ACCUM_STEPS = 2 # Effective Batch Size = 16 * 2 = 32

    NUM_WORKERS = 8
    TRAIN_SUBSET_FRACTION = 1.0
    TEST_SUBSET_FRACTION = 1.0
    EPOCHS = 30
    LR = 1e-5
    MAX_LEN = 128

    TEXT_MODEL = "microsoft/deberta-v3-large"
    IMAGE_MODEL = "openai/clip-vit-large-patch14"

    TRAIN_IMAGE_DIR = "../images_train"
    TEST_IMAGE_DIR = "../images_test"
    SEED = 42

    CHECKPOINT_DIR = "./checkpoints_deberta-v3-large_clip-large"
    
    LATEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pth")

print(f"Using device: {Config.DEVICE}")
print(f"Physical Batch Size: {Config.BATCH_SIZE}, Grad Accum Steps: {Config.GRAD_ACCUM_STEPS} --> Effective Batch Size: {Config.BATCH_SIZE * Config.GRAD_ACCUM_STEPS}")
print(f"Learning Rate: {Config.LR}")
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

Using device: cuda
Physical Batch Size: 16, Grad Accum Steps: 2 --> Effective Batch Size: 32
Learning Rate: 1e-05


In [ ]:
# Defined dataset parent folder
DATASET_FOLDER = '../dataset/'

# Load the datasets
train_df = pd.read_csv(DATASET_FOLDER + 'train.csv')
test_df = pd.read_csv(DATASET_FOLDER + 'test.csv')

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

Training data shape: (75000, 4)
Test data shape: (75000, 3)


In [ ]:
# Defined train images parent directory

IMAGE_TRAIN_FOLDER = '../images_train/' 
train_df['filename'] = train_df['image_link'].str.split('/').str[-1]
train_df.head()

,sample_id,catalog_content,image_link,price,filename
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,51mo8htwTHL.jpg
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,71YtriIHAAL.jpg
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,51+PFEe-w-L.jpg
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,41mu0HAToDL.jpg
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,41sA037+QvL.jpg


In [7]:
train_df['image_path'] = IMAGE_TRAIN_FOLDER + train_df['filename']

In [ ]:
# Defined test images parent directory

IMAGE_TEST_FOLDER = '../images_test/' 
test_df['filename'] = test_df['image_link'].str.split('/').str[-1]
test_df['image_path'] = IMAGE_TEST_FOLDER + test_df['filename']

In [ ]:
def parse_catalog_content(text):
    """
    Parses the structured text from catalog_content into separate features.
    """
    # Use a dictionary to store our extracted features
    features = {
        'item_name': '',
        'bullet_points': '',
        'value': np.nan, # Use NaN (Not a Number) for missing numerical values
        'unit': 'unknown'
    }

    # Ensure the input is a string
    text = str(text)

    # Look for "Item Name:" followed by any characters until the next "Bullet Point"
    match = re.search(r'Item Name:(.*?)(Bullet Point|$)', text, re.DOTALL)
    if match:
        features['item_name'] = match.group(1).strip()

    # Find all occurrences of "Bullet Point X:"
    bullets = re.findall(r'Bullet Point \d+:(.*?)(Bullet Point|Value|$)', text, re.DOTALL)
    # Join all found bullet points into a single string
    features['bullet_points'] = ' '.join([b[0].strip() for b in bullets])

    match = re.search(r'Value:\s*([\d\.]+)', text)
    if match:
        try:
            # Convert the extracted string to a float
            features['value'] = float(match.group(1))
        except ValueError:
            # If conversion fails, leave it as nan
            pass 

    # --- Extract Unit ---
    match = re.search(r'Unit:\s*(\w+)', text)
    if match:
        features['unit'] = match.group(1).strip()
        
    return features


def extract_ipq(text):
    """
    Extracts the Item Pack Quantity (IPQ) from a product description string.
    """
    text = str(text).lower()
    patterns = [
        r'pack of (\d+)', r'(\d+)\s*-?\s*pack', r'(\d+)\s*count',
        r'(\d+)\s*ct', r'ipq\s*:\s*(\d+)', r'case of (\d+)'
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return int(match.group(1))
    return 1 # Default to 1 if no pack size is found

In [ ]:
# Use our function to parse the catalog content for the training data

parsed_features_train = train_df['catalog_content'].apply(parse_catalog_content).apply(pd.Series)

parsed_features_test = test_df['catalog_content'].apply(parse_catalog_content).apply(pd.Series)

# Add the new parsed columns to our dataframes
train_df = pd.concat([train_df, parsed_features_train], axis=1)
test_df = pd.concat([test_df, parsed_features_test], axis=1)
train_df['ipq'] = train_df['catalog_content'].apply(extract_ipq)
test_df['ipq'] = test_df['catalog_content'].apply(extract_ipq)

In [ ]:
# Combine the item name and bullet points into a single 'clean_text' column

train_df['clean_text'] = train_df['item_name'] + ' ' + train_df['bullet_points']
test_df['clean_text'] = test_df['item_name'] + ' ' + test_df['bullet_points']
train_df[['catalog_content', 'clean_text']].head()

,catalog_content,clean_text
0,"Item Name: La Victoria Green Taco Sauce Mild, ...","La Victoria Green Taco Sauce Mild, 12 Ounce (P..."
1,"Item Name: Salerno Cookies, The Original Butte...","Salerno Cookies, The Original Butter Cookies, ..."
2,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...","Bear Creek Hearty Soup Bowl, Creamy Chicken wi..."
3,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...
4,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...","kedem Sherry Cooking Wine, 12.7 Ounce - 12 per..."


In [ ]:
# Remove the original columns that we've already processed

train_df_cleaned = train_df.drop(columns=['catalog_content', 'filename', 'item_name', 'bullet_points','image_link'])
test_df_cleaned = test_df.drop(columns=['catalog_content', 'filename', 'item_name', 'bullet_points','image_link'])

In [13]:
train_df_cleaned.head()

,sample_id,price,image_path,value,unit,ipq,clean_text
0,33127,4.89,../images_train/51mo8htwTHL.jpg,72.00,Fl,6,"La Victoria Green Taco Sauce Mild, 12 Ounce (P..."
1,198967,13.12,../images_train/71YtriIHAAL.jpg,32.00,Ounce,4,"Salerno Cookies, The Original Butter Cookies, ..."
2,261251,1.97,../images_train/51+PFEe-w-L.jpg,11.40,Ounce,6,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi..."
3,55858,30.34,../images_train/41mu0HAToDL.jpg,11.25,Ounce,1,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...
4,292686,66.49,../images_train/41sA037+QvL.jpg,12.00,Count,1,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per..."


In [ ]:
# Apply a log transformation to the price. This helps the model handle large price differences.

train_df_cleaned['log_price'] = np.log1p(train_df_cleaned['price'])

In [ ]:
# Find the median (middle) value of the 'value' column in the training data

median_value = train_df_cleaned['value'].median()
median_value_test=test_df_cleaned['value'].median()
# Fill missing values in both train and test sets with this median
train_df_cleaned['value'].fillna(median_value, inplace=True)
test_df_cleaned['value'].fillna(median_value, inplace=True)

In [16]:
train_df_cleaned.columns

Index(['sample_id', 'price', 'image_path', 'value', 'unit', 'ipq',
       'clean_text', 'log_price'],
      dtype='object')

In [17]:
test_df_cleaned.columns

Index(['sample_id', 'image_path', 'value', 'unit', 'ipq', 'clean_text'], dtype='object')

In [ ]:
def prepare_data(cleaned_df):
    df = cleaned_df.copy()

    # --- One-hot encode 'unit' ---
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    # Ensure the 'unit' column is a 2D array for the encoder
    encoded_units = encoder.fit_transform(df[["unit"]])
    encoded_df = pd.DataFrame(encoded_units, columns=encoder.get_feature_names_out(["unit"]))
    
    # Reset the index of BOTH DataFrames before joining to ensure perfect alignment.
    df_reset = df.reset_index(drop=True)
    df = pd.concat([df_reset, encoded_df], axis=1)
    
    df = df.drop(columns=["unit"])

    # --- Train/Validation Split ---
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=Config.SEED)

    # --- Scale ONLY the true numeric features ---
    scaler = StandardScaler()
    numeric_cols_to_scale = ["value", "ipq"]
    train_df[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
    val_df[numeric_cols_to_scale] = scaler.transform(val_df[numeric_cols_to_scale])

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        encoder,
        scaler,
    )

In [ ]:
import torch
from PIL import Image

# This class defines how our data is loaded for PyTorch
class ProductDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, image_processor, numeric_cols, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.processor = image_processor
        self.numeric_cols = numeric_cols
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    # This gets a single item (image, text, numbers) from our dataset
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = row["clean_text"]
        text_inputs = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=Config.MAX_LEN,
            return_tensors="pt"
        )

        try:
            img = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            # Create a blank image if the file is missing or corrupt
            img = Image.new("RGB", (224, 224), color=(255, 255, 255))

        # The CLIPProcessor will correctly resize, crop, and normalize the image
        img_inputs = self.processor(images=img, return_tensors="pt")

        numeric = torch.tensor(row[self.numeric_cols].to_numpy(dtype=np.float32))

        output = {
            "input_ids": text_inputs["input_ids"].squeeze(),
            "attention_mask": text_inputs["attention_mask"].squeeze(),
            "pixel_values": img_inputs["pixel_values"].squeeze(),
            "numeric": numeric
        }

        # If we're training, also include the price to check against
        if self.is_train:
            output["log_price"] = torch.tensor(row["log_price"], dtype=torch.float32)
            output["price"] = torch.tensor(row["price"], dtype=torch.float32)

        return output

In [20]:
def prepare_inference_data(new_df, encoder, scaler):
    df = new_df.copy()

    # Encode units using trained encoder
    encoded_units = encoder.transform(df[["unit"]])
    encoded_df = pd.DataFrame(encoded_units, columns=encoder.get_feature_names_out(["unit"]))
    df = pd.concat([df.drop(columns=["unit"]), encoded_df], axis=1)

    # Scale numeric features using trained scaler
    numeric_cols = ["value", "ipq"]
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    return df

In [ ]:
class MultiModalRegressor(nn.Module):
    # This defines the structure of our model
    def __init__(self, text_encoder, img_encoder, numeric_feature_size):
        super().__init__()
        self.text_encoder = text_encoder
        self.img_encoder = img_encoder

        text_hidden_size = text_encoder.config.hidden_size
        image_hidden_size = img_encoder.config.hidden_size

        self.text_proj = nn.Linear(text_hidden_size, 512)
        self.img_proj = nn.Linear(image_hidden_size, 512)
        self.num_proj = nn.Linear(numeric_feature_size, 128)

        fused_embedding_size = 512 + 512 + 128
        self.head = nn.Sequential(
            nn.Linear(fused_embedding_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    # This defines how data flows through the model
    def forward(self, batch):
        # get the last hidden state and average the tokens.
        txt_outputs = self.text_encoder(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        txt_out = txt_outputs.last_hidden_state.mean(dim=1)

        img_out = self.img_encoder(pixel_values=batch["pixel_values"]).pooler_output

        num_out = self.num_proj(batch["numeric"])

        fused = torch.cat([self.text_proj(txt_out), self.img_proj(img_out), num_out], dim=1)
        return self.head(fused)

In [22]:
def smape(y_true, y_pred):
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(num / (den + 1e-8)) * 100


In [ ]:
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPModel, get_linear_schedule_with_warmup

def train_model(train_df, val_df):
    # Model and Processor Loading
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    image_processor = CLIPProcessor.from_pretrained(Config.IMAGE_MODEL)
    text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
    img_encoder = CLIPModel.from_pretrained(Config.IMAGE_MODEL).vision_model
    
    numeric_cols = ['value', 'ipq'] + [c for c in train_df.columns if c.startswith("unit_")]
    numeric_feature_size = len(numeric_cols)
    
    model = MultiModalRegressor(text_encoder, img_encoder, numeric_feature_size).to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = nn.SmoothL1Loss()
    scaler = GradScaler()
    
    train_ds = ProductDataset(train_df, tokenizer, image_processor, numeric_cols, is_train=True)
    val_ds = ProductDataset(val_df, tokenizer, image_processor, numeric_cols, is_train=True)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

    num_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_steps)

    # START OF CHECKPOINTING CHANGES 
    
    best_smape = float('inf')
    start_epoch = 0

    # 1. Check if a checkpoint exists and load its state
    if os.path.exists(Config.LATEST_CHECKPOINT_PATH):
        print(f"Resuming training from checkpoint: {Config.LATEST_CHECKPOINT_PATH}")
        checkpoint = torch.load(Config.LATEST_CHECKPOINT_PATH)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        
        start_epoch = checkpoint['epoch'] + 1
        best_smape = checkpoint['best_smape']
        print(f"Resumed from Epoch {start_epoch}. Best SMAPE so far: {best_smape:.2f}%")
    else:
        print("Starting training from scratch.")

    # 2. Modify the loop to start from the correct epoch
    for epoch in range(start_epoch, Config.EPOCHS):
        model.train()
        total_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        for batch in pbar:
            y = batch.pop("log_price").to(Config.DEVICE).unsqueeze(1)
            batch.pop("price")
            batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            with autocast():
                preds = model(batch)
                loss = criterion(preds, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        
        
        model.eval()
        preds_all, trues_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                y_true = batch.pop("price").numpy()
                batch.pop("log_price")
                batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
                with autocast():
                    preds = model(batch).cpu().squeeze().numpy()
                
                preds_all.append(np.expm1(preds))
                trues_all.append(y_true)
        preds_all = np.concatenate(preds_all)
        trues_all = np.concatenate(trues_all)
        val_smape = smape(trues_all, preds_all)
        print(f"Epoch {epoch+1}: TrainLoss={avg_loss:.4f} | ValSMAPE={val_smape:.2f}%")

        # Saving the best model (logic is unchanged)
        if val_smape < best_smape:
            best_smape = val_smape
            # We add the epoch number to the filename for better tracking
            torch.save(model.state_dict(), os.path.join(Config.CHECKPOINT_DIR, f"best_model_epoch_{epoch+1}_{val_smape:.2f}.pth"))
            print(f"Saved new best model with SMAPE={val_smape:.2f}%")
        
        # 3. Save a full checkpoint after every epoch
        print(f"Saving latest checkpoint for epoch {epoch+1}...")
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_smape': best_smape
        }
        torch.save(checkpoint, Config.LATEST_CHECKPOINT_PATH)
        print("Checkpoint saved.")
        
    # END OF CHECKPOINTING CHANGES

    print(f"\nTraining done. Best SMAPE={best_smape:.2f}%")
    return model

In [ ]:
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPModel, get_linear_schedule_with_warmup

def train_model(train_df, val_df):
    # --- Model and Processor Loading (Unchanged) ---
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    image_processor = CLIPProcessor.from_pretrained(Config.IMAGE_MODEL)
    text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
    img_encoder = CLIPModel.from_pretrained(Config.IMAGE_MODEL).vision_model
    
    numeric_cols = ['value', 'ipq'] + [c for c in train_df.columns if c.startswith("unit_")]
    numeric_feature_size = len(numeric_cols)
    
    model = MultiModalRegressor(text_encoder, img_encoder, numeric_feature_size).to(Config.DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = nn.SmoothL1Loss()
    scaler = GradScaler()
    
    train_ds = ProductDataset(train_df, tokenizer, image_processor, numeric_cols, is_train=True)
    val_ds = ProductDataset(val_df, tokenizer, image_processor, numeric_cols, is_train=True)
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)

    num_steps = (len(train_loader) // Config.GRAD_ACCUM_STEPS) * Config.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_steps)

    best_smape = float('inf')
    start_epoch = 0

    if os.path.exists(Config.LATEST_CHECKPOINT_PATH):
        print(f"Resuming training from checkpoint: {Config.LATEST_CHECKPOINT_PATH}")
        checkpoint = torch.load(Config.LATEST_CHECKPOINT_PATH)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_smape = checkpoint['best_smape']
        print(f"Resumed from Epoch {start_epoch}. Best SMAPE so far: {best_smape:.2f}%")
    else:
        print("Starting training from scratch.")

    for epoch in range(start_epoch, Config.EPOCHS):
        model.train()
        total_loss = 0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        
        # --- FIX: MODIFIED TRAINING LOOP FOR GRADIENT ACCUMULATION ---
        for i, batch in pbar:
            y = batch.pop("log_price").to(Config.DEVICE).unsqueeze(1)
            batch.pop("price")
            batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
            
            with autocast():
                preds = model(batch)
                loss = criterion(preds, y)
                # Normalize the loss for accumulation
                loss = loss / Config.GRAD_ACCUM_STEPS
            
            # Accumulate gradients
            scaler.scale(loss).backward()
            
            # Perform weight update only after accumulating gradients for GRAD_ACCUM_STEPS
            if (i + 1) % Config.GRAD_ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                
            total_loss += loss.item() * Config.GRAD_ACCUM_STEPS
            pbar.set_postfix(loss=f"{loss.item() * Config.GRAD_ACCUM_STEPS:.4f}")

        avg_loss = total_loss / len(train_loader)
        
        # --- [Validation and Checkpointing logic remains the same] ---
        model.eval()
        preds_all, trues_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                y_true = batch.pop("price").numpy()
                batch.pop("log_price")
                batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
                with autocast():
                    preds = model(batch).cpu().squeeze().numpy()
                preds_all.append(np.expm1(preds))
                trues_all.append(y_true)
        preds_all = np.concatenate(preds_all)
        trues_all = np.concatenate(trues_all)
        val_smape = smape(trues_all, preds_all)
        print(f"Epoch {epoch+1}: TrainLoss={avg_loss:.4f} | ValSMAPE={val_smape:.2f}%")

        if val_smape < best_smape:
            best_smape = val_smape
            torch.save(model.state_dict(), os.path.join(Config.CHECKPOINT_DIR, f"best_model_epoch_{epoch+1}_{val_smape:.2f}.pth"))
            print(f"Saved new best model with SMAPE={val_smape:.2f}%")
        
        print(f"Saving latest checkpoint for epoch {epoch+1}...")
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(), 'best_smape': best_smape
        }
        torch.save(checkpoint, Config.LATEST_CHECKPOINT_PATH)
        print("Checkpoint saved.")
        
    print(f"\nTraining done. Best SMAPE={best_smape:.2f}%")
    return model

In [24]:
# --- Step 1: Create a smaller sample of your data ---
# n_samples = 25000
# print(f"Creating a random subset of {n_samples} images for training...")
# train_subset_df = train_df_cleaned.sample(n=n_samples, random_state=Config.SEED)
# print("Preparing the data subset...")
train_df_subset, val_df_subset, encoder, scaler = prepare_data(train_df_cleaned)


In [25]:
train_df_subset = train_df_subset.replace([np.inf, -np.inf], np.nan).dropna()


In [26]:
train_df_subset['clean_text'].isnull().sum()

0

In [27]:
import torch

if torch.cuda.is_available():
    print("Clearing PyTorch CUDA cache...")
    torch.cuda.empty_cache()
    print("Cache cleared.")

Clearing PyTorch CUDA cache...
Cache cleared.


In [ ]:
#call our function to start training the model!
trained_model = train_model(train_df_subset, val_df_subset)

/usr/local/lib/python3.8/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Resuming training from checkpoint: ./checkpoints_deberta-v3-large_clip-large/latest_checkpoint.pth


Error during conversion: ChunkedEncodingError(ProtocolError("Connection broken: InvalidChunkLength(got length b'', 0 bytes read)", InvalidChunkLength(got length b'', 0 bytes read)))


Resumed from Epoch 20. Best SMAPE so far: 44.21%


Epoch 21/30: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4219/4219 [19:50<00:00,  3.54it/s, loss=0.0076]


...

Epoch 28: TrainLoss=0.0110 | ValSMAPE=44.22%
Saving latest checkpoint for epoch 28...
Checkpoint saved.


Epoch 29/30: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4219/4219 [20:00<00:00,  3.51it/s, loss=0.0082]


Epoch 29: TrainLoss=0.0106 | ValSMAPE=44.06%
Saved new best model with SMAPE=44.06%
Saving latest checkpoint for epoch 29...
Checkpoint saved.


Epoch 30/30: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 4219/4219 [25:27<00:00,  2.76it/s, loss=0.0208]


Epoch 30: TrainLoss=0.0101 | ValSMAPE=44.12%
Saving latest checkpoint for epoch 30...
Checkpoint saved.

Training done. Best SMAPE=44.06%


In [ ]:
# Check types of all columns
print(train_df.dtypes)

# Check only numeric columns
numeric_cols = ["value", "ipq"]  # your numeric features
print(train_df[numeric_cols].dtypes)


In [45]:
train_df.columns

Index(['sample_id', 'price', 'image_path', 'value', 'ipq', 'clean_text',
       'log_price', 'unit_1', 'unit_2', 'unit_20', 'unit_24', 'unit_7',
       'unit_8', 'unit_BOX', 'unit_Bag', 'unit_Bottle', 'unit_Box',
       'unit_Bucket', 'unit_CASE', 'unit_COUNT', 'unit_CT', 'unit_Can',
       'unit_Carton', 'unit_Comes', 'unit_Count', 'unit_Each', 'unit_FL',
       'unit_Fl', 'unit_Fluid', 'unit_Foot', 'unit_Gram', 'unit_Grams',
       'unit_Jar', 'unit_K', 'unit_LB', 'unit_Liters', 'unit_None', 'unit_OZ',
       'unit_Ounce', 'unit_Ounces', 'unit_Oz', 'unit_PACK', 'unit_Pack',
       'unit_Packs', 'unit_Paper', 'unit_Per', 'unit_Piece', 'unit_Pouch',
       'unit_Pound', 'unit_Pounds', 'unit_Sq', 'unit_Tea', 'unit_Ziplock',
       'unit_bag', 'unit_bottle', 'unit_bottles', 'unit_box', 'unit_can',
       'unit_capsule', 'unit_cm', 'unit_count', 'unit_ct', 'unit_each',
       'unit_fl', 'unit_fluid', 'unit_gr', 'unit_gram', 'unit_gramm',
       'unit_grams', 'unit_in', 'unit_kg', 'unit_lb

In [48]:
numeric_cols = ["value", "ipq"] + [col for col in train_df.columns if col.startswith("unit_")]
print(train_df[numeric_cols].dtypes.unique())

[dtype('float64')]


In [59]:
print(set(numeric_cols) - set(train_df.columns))  # Should be empty


set()


In [30]:
test_df_cleaned.head()

,sample_id,image_path,value,unit,ipq,clean_text
0,100179,../images_test/71hoAn78AWL.jpg,10.5,Ounce,1,Rani 14-Spice Eshamaya's Mango Chutney (Indian...
1,245611,../images_test/61ex8NHCIjL.jpg,2.0,Fl,1,Natural MILK TEA Flavoring extract by HALO PAN...
2,146263,../images_test/61KCM61J8eL.jpg,32.0,Ounce,1,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...
3,95658,../images_test/51Ex6uOH7yL.jpg,2.0,Count,2,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2...
4,36806,../images_test/71QYlrOMoSL.jpg,32.0,Fl,1,"McCormick Culinary Vanilla Extract, 32 fl oz -..."


In [27]:
test_final_df = prepare_inference_data(test_df_cleaned, encoder, scaler)

In [28]:
# --- Step 2: Initialize the Models and Processors ---
# These must match the models used during training.
print("Loading model components...")
tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
image_processor = CLIPProcessor.from_pretrained(Config.IMAGE_MODEL)
text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
img_encoder = CLIPModel.from_pretrained(Config.IMAGE_MODEL).vision_model

Loading model components...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
numeric_cols = ['value', 'ipq'] + [c for c in train_df.columns if c.startswith("unit_")]
numeric_feature_size = len(numeric_cols)
# -------------------------------------------

# Re-create the model structure
final_model = MultiModalRegressor(text_encoder, img_encoder, numeric_feature_size).to(Config.DEVICE)

# Define the path to your best saved model checkpoint
# **IMPORTANT**: Update the filename to match the best score you got (e.g., 48.39)
best_model_path = os.path.join(Config.CHECKPOINT_DIR, "best_model_45.55.pth")

print(f"Loading best model from: {best_model_path}")

# Load the saved weights into the model structure
final_model.load_state_dict(torch.load(best_model_path))
final_model.eval() # Set the model to evaluation mode (very important!)

# --- Step 4: Create the Test Dataset and DataLoader ---

# Get the list of all numeric and one-hot encoded columns
# numeric_cols = ['value', 'ipq'] + [c for c in test_final_df.columns if c.startswith("unit_")]


In [75]:
# Create the dataset for the test data
test_ds = ProductDataset(
    test_final_df,
    tokenizer,
    image_processor,
    numeric_cols,
    is_train=False  # Set is_train to False
)

# Create a DataLoader to process the test data in batches
# A larger batch size can be used for inference
test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=Config.BATCH_SIZE * 2, # Often can double batch size for inference
    shuffle=False,
    num_workers=Config.NUM_WORKERS
)


In [ ]:
# --- Step 5: Run Predictions ---

test_preds = []
print("\nGenerating predictions on the test set...")

# No need to calculate gradients for inference
with torch.no_grad():
    for batch in tqdm(test_loader):
        # Move the batch of data to the GPU
        batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}

        # Get the model's predictions (they will be on the log scale)
        with autocast():
             preds_log = final_model(batch)

        # Move predictions to the CPU and store them
        test_preds.append(preds_log.cpu().squeeze().numpy())

# Concatenate predictions from all batches into a single array
final_predictions_log = np.concatenate(test_preds)


# --- Step 6: Format and Save the Submission File ---

# Convert predictions from log scale back to the original price scale
final_predictions = np.expm1(final_predictions_log)

# Create the submission DataFrame
submission_df = pd.DataFrame({
    'sample_id': test_final_df['sample_id'],
    'price': final_predictions
})

# Safety check: ensure no prices are negative
submission_df['price'] = submission_df['price'].clip(lower=0)

# Save to CSV
submission_df.to_csv('submission_44.06_model.csv', index=False)

print("\nSubmission file 'submission_dl_model.csv' created successfully!")
print("Here's a preview:")
print(submission_df.head())


Generating predictions on the test set...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 586/586 [14:21<00:00,  1.47s/it]



✅ Submission file 'submission_dl_model.csv' created successfully!
Here's a preview:
   sample_id      price
0     100179  34.875000
1     245611  13.898438
2     146263  25.093750
3      95658   4.941406
4      36806  18.734375


In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModel, CLIPProcessor
from sklearn.preprocessing import OneHotEncoder

# =====================================================================================
# CONFIGURATION - Must match the training configuration of the saved model
# =====================================================================================
class Config:
    TEXT_MODEL = "microsoft/deberta-v3-large"
    IMAGE_MODEL = "openai/clip-vit-large-patch14"
    
    # !!! USER ACTION: VERIFY THESE PATHS ARE CORRECT !!!
    CHECKPOINT_DIR = "./checkpoints_deberta-v3-large_clip-large"
    BEST_MODEL_FILE = "best_model_epoch_29_44.06.pth" 
    
    DATASET_FOLDER = '../dataset/'
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================================
# HELPER FUNCTIONS & CLASSES (Copied from your notebook for consistency)
# =====================================================================================

def parse_catalog_content(text):
    features = {'item_name': '', 'bullet_points': '', 'value': np.nan, 'unit': 'unknown'}
    text = str(text)
    match = re.search(r'Item Name:(.*?)(Bullet Point|$)', text, re.DOTALL)
    if match: features['item_name'] = match.group(1).strip()
    bullets = re.findall(r'Bullet Point \d+:(.*?)(Bullet Point|Value|$)', text, re.DOTALL)
    features['bullet_points'] = ' '.join([b[0].strip() for b in bullets])
    return features

class MultiModalRegressor(nn.Module):
    def __init__(self, text_encoder, img_encoder, numeric_feature_size):
        super().__init__()
        self.text_encoder = text_encoder
        self.img_encoder = img_encoder
        text_hidden_size = text_encoder.config.hidden_size
        image_hidden_size = img_encoder.config.hidden_size
        self.text_proj = nn.Linear(text_hidden_size, 512)
        self.img_proj = nn.Linear(image_hidden_size, 512)
        self.num_proj = nn.Linear(numeric_feature_size, 128)
        fused_embedding_size = 512 + 512 + 128
        self.head = nn.Sequential(nn.Linear(fused_embedding_size, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, 1))
    def forward(self, batch):
        txt_outputs = self.text_encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        if hasattr(txt_outputs, 'pooler_output') and txt_outputs.pooler_output is not None:
             txt_out = txt_outputs.pooler_output
        else:
             txt_out = txt_outputs.last_hidden_state.mean(dim=1)
        img_out = self.img_encoder(pixel_values=batch["pixel_values"]).pooler_output
        num_out = self.num_proj(batch["numeric"])
        fused = torch.cat([self.text_proj(txt_out), self.img_proj(img_out), num_out], dim=1)
        return self.head(fused)

# =====================================================================================
# MAIN PARAMETER COUNTING SCRIPT
# =====================================================================================

# --- 1. Recreate the exact number of numeric features ---
print("Step 1: Determining numeric feature size from the original data...")
try:
    train_df = pd.read_csv(Config.DATASET_FOLDER + 'train.csv')
    parsed_features_train = train_df['catalog_content'].apply(parse_catalog_content).apply(pd.Series)
    train_df = pd.concat([train_df, parsed_features_train], axis=1)

    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    encoder.fit(train_df[["unit"]])
    one_hot_cols = encoder.get_feature_names_out(["unit"])

    # The numeric_feature_size is 'value' + 'ipq' + all the one-hot encoded 'unit' columns
    numeric_feature_size = 2 + len(one_hot_cols)
    print(f"Determined that the model was trained with {numeric_feature_size} numeric features.")
except FileNotFoundError:
    print("Could not find train.csv. Please ensure the path is correct to determine the numeric feature size.")
    numeric_feature_size = 85 # Fallback to the known value from your notebook if file not found

# --- 2. Build the model scaffold ---
print("\nStep 2: Building the model architecture...")
text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
img_encoder = AutoModel.from_pretrained(Config.IMAGE_MODEL).vision_model
model_scaffold = MultiModalRegressor(text_encoder, img_encoder, numeric_feature_size)

# --- 3. Load the saved weights (to verify architecture) ---
best_model_path = os.path.join(Config.CHECKPOINT_DIR, Config.BEST_MODEL_FILE)
print(f"Step 3: Verifying architecture by loading saved weights from {best_model_path}...")
try:
    model_scaffold.load_state_dict(torch.load(best_model_path, map_location='cpu')) # Load to CPU to save GPU memory
    print("Weights loaded successfully. Architecture is correct.")
except Exception as e:
    print(f"Warning: Could not load weights. Error: {e}. Counting parameters of the architecture only.")
    
# --- 4. Count the parameters ---
print("\nStep 4: Counting parameters...")
total_params = sum(p.numel() for p in model_scaffold.parameters())
trainable_params = sum(p.numel() for p in model_scaffold.parameters() if p.requires_grad)

print("\n" + "="*50)
print("           MODEL PARAMETER COUNT          ")
print("="*50)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Total (in millions):  {total_params / 1_000_000:.2f}M")
print("="*50)

### Step 1: Determining numeric feature size from the original data...
Determined that the model was trained with 3 numeric features.

### Step 2: Building the model architecture...

### Step 3: Verifying architecture by loading saved weights from ./checkpoints_deberta-v3-large_clip-large/best_model_epoch_29_44.06.pth...

### Step 4: Counting parameters...

## MODEL PARAMETER COUNT   

- Total Parameters:     738,832,897 
- Trainable Parameters: 738,832,897 
- Total (in millions):  738.83M 

In [ ]:
import pandas as pd
import numpy as np

# --- 1. SETUP: Define your file paths ---

# Validation files (predictions made on your local validation set)
# These files MUST have 'sample_id', 'true_price', and 'predicted_price' columns
val_preds_path_A = 'validation_predictions_48.23.csv' # Your first model
val_preds_path_B = 'validation_predictions_45.55.csv' # Your second model
val_preds_path_C = 'validation_predictions_44.06.csv' # Your best model

# Test files (final submissions for the official test set)
test_preds_path_A = 'submission_A_47.25.csv'
test_preds_path_B = 'submission_B_44.60.csv'
test_preds_path_C = 'submission_C_43.41.csv'


# --- 2. LOAD AND VALIDATE DATA ---

print("Loading validation prediction files...")
df_A = pd.read_csv(val_preds_path_A)
df_B = pd.read_csv(val_preds_path_B)
df_C = pd.read_csv(val_preds_path_C)



# --- Sanity Check: Make sure all IDs match ---
if not (df_A['sample_id'].equals(df_B['sample_id']) and df_A['sample_id'].equals(df_C['sample_id'])):
    raise ValueError("CRITICAL ERROR: The 'sample_id' columns in your validation files do not match. Aborting.")
else:
    print("✅ Sanity check passed: All sample_ids are perfectly aligned.")

# Extract the prediction and true value arrays
preds_A = df_A['predicted_price'].values
preds_B = df_B['predicted_price'].values
preds_C = df_C['predicted_price'].values
y_true_validation = df_A['true_price'].values


# --- 3. FIND OPTIMAL WEIGHTS ---

def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    epsilon = 1e-8
    return np.mean(numerator / (denominator + epsilon)) * 100

best_smape = 100.0
best_weights = (0, 0,0)

print("\nSearching for the best blending weights...")
# Iterate through different weight combinations
for w_a in np.arange(0, 0.20, 0.05):      # Weight for model A (worst) from 0% to 30%
    for w_b in np.arange(0, 0.70, 0.05):  # Weight for model B (middle) from 0% to 50%
        w_c = 1.0 - w_a - w_b            # The rest of the weight goes to model C (best)

        # Skip invalid weight combinations
        if w_c < 0:
            continue

        # Calculate the weighted average for the validation set
        ensemble_pred = (w_a * preds_A) + (w_b * preds_B) + (w_c * preds_C)

        # Calculate the SMAPE score for this combination
        current_smape = smape(y_true_validation, ensemble_pred)

        # If this is the best score so far, save it
        if current_smape < best_smape:
            best_smape = current_smape
            best_weights = (w_a, w_b, w_c)
            print(f"New best SMAPE: {best_smape:.4f}% with weights A={w_a:.2f}, B={w_b:.2f}, C={w_c:.2f}")

print("\n--- Search Complete ---")
print(f"🏆 Best Validation SMAPE: {best_smape:.4f}%")
print(f"Optimal Weights (A, B, C): {best_weights}")


